## Object Detection with TensorFlow 2

This notebook demonstrates a real-time object detection application using a pre-trained TensorFlow 2 model and a webcam feed. It leverages the TensorFlow Object Detection API for building, training, and deploying object detection models.

### 1. Import Libraries

This cell imports all the necessary libraries for the object detection pipeline, including TensorFlow, NumPy, Matplotlib, OpenCV, and various modules from the TensorFlow Object Detection API.

In [1]:
import os
import sys


import numpy as np
import tensorflow as tf
from matplotlib import pyplot as plt
from google.protobuf import text_format

import cv2

from object_detection.utils import config_util
from object_detection.utils import label_map_util
from object_detection.utils import visualization_utils as viz_utils
from object_detection.builders import model_builder
from object_detection.protos import pipeline_pb2

C:\Users\DELL\.conda\envs\robot\lib\site-packages\google\api_core\_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.20) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


### 2. Define Paths

This cell defines the file system paths for various components of the object detection project, such as the workspace, scripts, API models, annotations, images, trained models, pre-trained models, and configuration files. These paths are crucial for organizing the project structure and accessing resources.

In [2]:
WORKSPACE_PATH = 'Tensorflow/workspace'
SCRIPTS_PATH = 'Tensorflow/scripts'
APIMODEL_PATH = 'Tensorflow/models'
ANNOTATION_PATH = WORKSPACE_PATH+'/annotations'
IMAGE_PATH = WORKSPACE_PATH+'/images'
MODEL_PATH = WORKSPACE_PATH+'/models'
PRETRAINED_MODEL_PATH = WORKSPACE_PATH+'/pre-trained-models'
CONFIG_PATH = MODEL_PATH+'/my_ssd_mobnet/pipeline.config'
CHECKPOINT_PATH = MODEL_PATH+'/my_ssd_mobnet/'

### 3. Define Custom Model Name

This cell sets the name for the custom object detection model. This name is used to identify the model's configuration and checkpoint files within the project structure.

In [3]:
CUSTOM_MODEL_NAME = 'my_ssd_mobnet'

### 4. Load Model Configuration

This cell constructs the full path to the `pipeline.config` file for the custom model and then uses `config_util.get_configs_from_pipeline_file` to load the model's configuration. This configuration contains all the parameters defining the model's architecture and training setup.

In [4]:
CONFIG_PATH = MODEL_PATH + '/' + CUSTOM_MODEL_NAME + '/pipeline.config'
config = config_util.get_configs_from_pipeline_file(CONFIG_PATH)

### 5. Build and Restore Detection Model

This cell builds the object detection model using the loaded configuration (`configs['model']`) and sets it to inference mode (`is_training=False`). It then restores the model's weights from a pre-trained checkpoint, ensuring the model is ready for detection.

In [5]:
configs = config_util.get_configs_from_pipeline_file(CONFIG_PATH)
detection_model = model_builder.build(model_config=configs['model'], is_training=False)

ckpt = tf.compat.v2.train.Checkpoint(model=detection_model)
ckpt.restore(os.path.join(CHECKPOINT_PATH, 'ckpt-6')).expect_partial()

### 6. Define Detection Function

This cell defines the `detect_fn` function, which encapsulates the entire detection process. This function takes an image tensor as input, preprocesses it, runs predictions through the model, and then post-processes the raw model output into human-readable detections (bounding boxes, classes, and scores). The `@tf.function` decorator optimizes the function for TensorFlow's graph execution.

In [6]:
@tf.function
def detect_fn(image):
    image, shapes = detection_model.preprocess(image)
    prediction_dict = detection_model.predict(image, shapes)
    detections = detection_model.postprocess(prediction_dict, shapes)
    return detections

### 7. Load Category Index

This cell loads the label map from `label_map.pbtxt` and creates a `category_index`. The `category_index` maps integer class IDs to human-readable labels, which is essential for visualizing the detection results.

In [7]:
category_index = label_map_util.create_category_index_from_labelmap(ANNOTATION_PATH+'/label_map.pbtxt')

### 8. Real-Time Object Detection Loop

This is the core of the real-time detection application. It performs the following steps:

1.  **Camera Setup**: Initializes the webcam (`cv2.VideoCapture(0)`) and sets the desired frame width and height.
2.  **Detection Loop**: Continuously captures frames from the webcam.
3.  **Preprocessing**: Converts the captured frame into a TensorFlow tensor and expands its dimensions.
4.  **Prediction**: Calls the `detect_fn` to get object detections for the current frame.
5.  **Extraction**: Extracts the bounding boxes, classes, and scores from the detection results.
6.  **Non-Max Suppression (NMS)**: Applies NMS to filter out overlapping bounding boxes, keeping only the most confident detections.
7.  **Visualization**: Draws the filtered bounding boxes and labels onto the original frame using `viz_utils.visualize_boxes_and_labels_on_image_array`.
8.  **Display**: Shows the processed frame in a window.
9.  **Exit Condition**: Breaks the loop and exits when the 'q' key is pressed.

Finally, it performs **Resource Cleanup** by releasing the camera and destroying all OpenCV windows.

In [8]:
# ==========================================
# 1. SETUP CAMERAS & PARAMETERS
# ==========================================
cap = cv2.VideoCapture(0)

# Request a larger display resolution from your hardware
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)

# The standard label offset for custom object detection models
label_id_offset = 1

print("Starting Real-Time Object Detection Window... Press 'q' to exit.")

# ==========================================
# 2. RUN REAL-TIME DETECTION LOOP
# ==========================================
while True:
    ret, frame = cap.read()
    if not ret:
        print("Error: Failed to grab frame from webcam.")
        break

    # Expand dimensions directly (OpenCV frames are already numpy arrays)
    input_tensor = tf.convert_to_tensor(np.expand_dims(frame, 0), dtype=tf.float32)

    # Run prediction through your trained model
    detections = detect_fn(input_tensor)

    # Extract total number of predictions detected in this frame
    num_detections = int(detections['num_detections'][0])

    # Pull raw prediction matrices and slice down to active detections
    boxes = detections['detection_boxes'][0, :num_detections].numpy()
    classes = detections['detection_classes'][0, :num_detections].numpy().astype(np.int64) + label_id_offset
    scores = detections['detection_scores'][0, :num_detections].numpy()

    # -----------------------------------------------------------------
    # FIX MULTIPLE BOXES: Non-Max Suppression (NMS) Filter
    # -----------------------------------------------------------------
    # max_output_size: limit to max 5 items on screen at once
    # iou_threshold: if boxes overlap more than 40%, discard the weaker prediction
    selected_indices = tf.image.non_max_suppression(
        boxes,
        scores,
        max_output_size=5,
        iou_threshold=0.40
    ).numpy()

    # Filter our prediction vectors using only the clean NMS indices
    boxes = boxes[selected_indices]
    classes = classes[selected_indices]
    scores = scores[selected_indices]
    # -----------------------------------------------------------------

    # Draw the filtered bounding boxes and label flags directly onto the image
    viz_utils.visualize_boxes_and_labels_on_image_array(
        frame,
        boxes,
        classes,
        scores,
        category_index,
        use_normalized_coordinates=True,
        max_boxes_to_draw=5,
        min_score_thresh=0.35,  # Adjusted to surface lower-confidence patterns from 200 images
        agnostic_mode=False
    )

    # Display window configurations with dynamic aspect ratio resizing
    cv2.namedWindow('object detection', cv2.WINDOW_NORMAL)
    cv2.imshow('object detection', frame)

    # Clean exit strategy mapping to the 'q' key
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# ==========================================
# 3. RESOURCE CLEANUP
# ==========================================
cap.release()
cv2.destroyAllWindows()
print("Webcam session closed safely.")

Starting Real-Time Object Detection Window... Press 'q' to exit.
Webcam session closed safely.
